# 相手の未公開情報を推定する

TreeSearchPlayer.estimate_opponent() をオーバーライドして、相手ポケモンの見えていない技を推定値で補完する。

In [1]:
%pip install -q jpoke

Note: you may need to restart the kernel to use updated packages.


In [2]:
from jpoke import Battle
from jpoke.players import RandomPlayer, TreeSearchPlayer

In [3]:
class OpponentGuessingPlayer(TreeSearchPlayer):
    """相手の未公開の技を「みずでっぽう」1本と仮定して探索するAI（estimate_opponent()の拡張例）。"""

    def estimate_opponent(self, battle: Battle) -> None:
        # 実戦では対戦データベースや使用率統計等から推定するが、ここでは固定の
        # 推測技で最小例を示す。opponent.active.moves が空（未公開）のときだけ
        # 書き込む
        opponent = battle.get_opponent(self)
        opponent_active = battle.get_active(opponent)
        if not opponent_active.moves:
            opponent_active.set_moves(["ハイドロポンプ"])

In [4]:
ai_player = OpponentGuessingPlayer("GuessingAI", max_plies=1, max_nodes=50)
ai_player.add_pokemon("リザードン", move_names=["かえんほうしゃ", "じしん"])

opponent_player = RandomPlayer("Opponent")
opponent_player.add_pokemon("カメックス", move_names=["ハイドロポンプ"])

battle = Battle(ai_player, opponent_player, seed=1)
battle.start()

# 1ターン目: 相手の技は未公開。estimate_opponent が推定を書き込むため
# fallback() には委譲されず、推定した相手の技も踏まえた探索が行われる
# （nodes_expanded > 0 になる。何も推定しない既定実装のままだと0のまま）
battle.step()
print(f"1ターン目に展開したノード数: {ai_player.nodes_expanded}")
battle.print_logs()

TypeError: OpponentGuessingPlayer.estimate_opponent() takes 2 positional arguments but 3 were given